```markdown
# Tutorial 9: Object Detection using SSD & YOLO

This notebook covers:
1. Installing dependencies.
2. Training a pretrained YOLO model on custom data.
3. Implementing/using a pretrained SSD model.
```

In [ ]:
# Install Ultralytics for YOLOv8/v9/v10 support
!pip install ultralytics -q

```markdown
## 1. YOLO (You Only Look Once) - Pretrained
YOLOv8. need a `data.yaml` file that points to training and validation images.
```

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLOv8n (nano) model - good for mobile/efficiency
model_yolo = YOLO('yolov8n.pt')

# Train the model on your custom data
# results = model_yolo.train(data='path/to/your/data.yaml', epochs=50, imgsz=640)

print("YOLO model loaded. Uncomment the train line once your data.yaml is ready.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO model loaded. Uncomment the train line once your data.yaml is ready.


```markdown
## 2. SSD (Single Shot MultiBox Detector) - Pretrained
`torchvision` to quickly grab a pretrained SSD model with a MobileNetV3 backbone.
```

In [ ]:
import torch
import torchvision
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights

# Load a pretrained SSD model
weights = SSD300_VGG16_Weights.DEFAULT
model_ssd = ssd300_vgg16(weights=weights)
model_ssd.eval() # Set to evaluation mode

print("SSD VGG16 model loaded.")

Downloading: "https://download.pytorch.org/models/ssd300_vgg16_coco-b556d3b4.pth" to /root/.cache/torch/hub/checkpoints/ssd300_vgg16_coco-b556d3b4.pth


100%|██████████| 136M/136M [00:01<00:00, 129MB/s]


SSD VGG16 model loaded.


## 3. Implementing SSD Architecture from Scratch (Simplified)
To understand the backbone and detection head, define a simplified version using PyTorch. This allows to add or remove layers to see how it affects the model.

In [ ]:
import torch
import torch.nn as nn

class SimpleSSD(nn.Module):
    def __init__(self, num_classes):
        super(SimpleSSD, self).__init__()
        # 1. Backbone: Feature Extractor
        # You can add more Conv2d layers here to improve complexity
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # 2. Detection Head: Predicts bounding box offsets and class scores
        # Number of anchors per location = 4
        self.classification_head = nn.Conv2d(32, num_classes * 4, kernel_size=3, padding=1)
        self.regression_head = nn.Conv2d(32, 4 * 4, kernel_size=3, padding=1)

    def forward(self, x):
        features = self.backbone(x)
        cls_scores = self.classification_head(features)
        bbox_regs = self.regression_head(features)
        return cls_scores, bbox_regs

# Instantiate a custom model
custom_ssd = SimpleSSD(num_classes=2)
print("Custom SSD-like model initialized:")
print(custom_ssd)

Custom SSD-like model initialized:
SimpleSSD(
  (backbone): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classification_head): Conv2d(32, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (regression_head): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)


## 4. Fine-tuning YOLO on Custom Data
`data.yaml` from the previous tutorial to train the model.

In [ ]:
# Example command to start training YOLOv8
# Replace 'path/to/data.yaml' with your actual file path
# model_yolo.train(data='path/to/data.yaml', epochs=10, imgsz=640)

print("Ready for YOLO training. Update the path when your data is ready.")

Ready for YOLO training. Update the path when your data is ready.


## 3. Implementing SSD Architecture from Scratch (Simplified)
To understand the backbone and detection head, define a simplified version using PyTorch layers. This allows to add or remove layers as requested.

In [ ]:
import torch
import torch.nn as nn

class SimpleSSD(nn.Module):
    def __init__(self, num_classes):
        super(SimpleSSD, self).__init__()
        # 1. Backbone: Feature Extractor (e.g., simplified VGG or MobileNet block)
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )

        # 2. Detection Head: Predicts bounding box offsets and class scores
        # You can add/remove layers here to see the effect on performance
        self.classification_head = nn.Conv2d(32, num_classes * 4, kernel_size=3, padding=1)
        self.regression_head = nn.Conv2d(32, 4 * 4, kernel_size=3, padding=1) # 4 boxes per cell

    def forward(self, x):
        features = self.backbone(x)
        cls_scores = self.classification_head(features)
        bbox_regs = self.regression_head(features)
        return cls_scores, bbox_regs

# Instantiate a custom model with a specific number of classes
custom_model = SimpleSSD(num_classes=2)
print("Custom simplified SSD model initialized.")
print(custom_model)

Custom simplified SSD model initialized.
SimpleSSD(
  (backbone): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classification_head): Conv2d(32, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (regression_head): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
)


## 4. Fine-tuning the Pretrained YOLO Model
 `data.yaml` ready from the previous tutorial,the following cell to start training. This fulfills the task of training on own data.

In [ ]:
# Example command to start training YOLOv8
# model_yolo.train(data='your_dataset.yaml', epochs=10, imgsz=640)
print("Ready for training. Update the data path in the line above when your dataset is mapped.")

Ready for training. Update the data path in the line above when your dataset is mapped.


## 3. Customizing the Models (Adding/Removing Layers)
Modify the detection heads of models to suit custom data requirements or experiment with different architectures.

In [ ]:
import torch.nn as nn

# Example: Modifying the SSD head
# Accessing the predictor layers in the SSD model
num_classes = 5  # Example: Change this to your number of classes
in_channels = model_ssd.head.classification_head.module_list[0].in_channels

# We can replace parts of the head if we want to change the number of outputs
print(f"Current SSD classification head info: {model_ssd.head.classification_head}")

# To 'remove' or 'add' layers from scratch, you'd typically define a class inheriting nn.Module
class CustomSSDHead(nn.Module):
    def __init__(self, original_head):
        super().__init__()
        self.module_list = original_head.module_list
        # Add a custom layer or modify existing ones here

    def forward(self, x):
        return self.module_list(x)

print("Head modification template ready.")

Current SSD classification head info: SSDClassificationHead(
  (module_list): ModuleList(
    (0): Conv2d(512, 364, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): Conv2d(1024, 546, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (2): Conv2d(512, 546, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): Conv2d(256, 546, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4-5): 2 x Conv2d(256, 364, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  )
)
Head modification template ready.
